# Reproducible sealed perf-corpus replay

Run this output-free notebook only inside an OS-enforced network-denied sandbox with an allowlisted environment, a complete read-only sealed-input and trusted-code closure, and a nonexistent bounded output directory. The committed manifest supplies trust anchors; caller-selected paths do not. A post-commit external receipt authenticates the publication manifest itself without self-reference.


In [ ]:
import hashlib
import json
import os
import pathlib

def fail(message):
    raise RuntimeError(message)

def require(condition, message):
    if not condition:
        fail(message)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def reject_symlink_components(path):
    current = pathlib.Path(path.anchor)
    for part in path.parts[1:]:
        current = current / part
        if current.exists() and current.is_symlink():
            fail(f"symlink component rejected: {current}")

control_flags = ("GJC_PERF_CORPUS_NETWORK_DENIED", "GJC_PERF_CORPUS_SANITIZED_ENV", "GJC_PERF_CORPUS_INPUT_MOUNT_READ_ONLY")
require(all(os.environ.get(name) == "1" for name in control_flags), "external sandbox controls are required")
publication_dir = pathlib.Path(os.environ["GJC_PERF_CORPUS_PUBLICATION_DIR"]).absolute()
bundle_dir = pathlib.Path(os.environ["GJC_PERF_CORPUS_BUNDLE_DIR"]).absolute()
input_dir = pathlib.Path(os.environ["GJC_PERF_CORPUS_INPUT_DIR"]).absolute()
output_dir = pathlib.Path(os.environ["GJC_PERF_CORPUS_OUTPUT_DIR"]).absolute()
for path in (publication_dir, bundle_dir, input_dir, output_dir.parent):
    reject_symlink_components(path)
require(not output_dir.exists(), "output directory must not exist")
output_dir.mkdir(mode=0o700)
manifest_path = publication_dir / "artifacts/perf-corpus-memory-evidence-manifest.json"
report_path = publication_dir / "artifacts/perf-corpus-memory-evidence-report.json"
notebook_path = publication_dir / "artifacts/perf-corpus-memory-evidence-notebook.ipynb"
manifest = json.loads(manifest_path.read_bytes())
require(set(manifest) == {"schema", "generatedAt", "trustedBindings", "artifacts", "publicationHeadBinding", "executionControls", "retentionAccess"}, "unexpected publication manifest keys")
anchors = manifest["trustedBindings"]
artifact_by_path = {item["path"]: item for item in manifest["artifacts"]}
require(set(artifact_by_path) == {"artifacts/perf-corpus-memory-evidence-report.json", "artifacts/perf-corpus-memory-evidence-notebook.ipynb"}, "unexpected publication artifact set")
for path, relative in ((report_path, "artifacts/perf-corpus-memory-evidence-report.json"), (notebook_path, "artifacts/perf-corpus-memory-evidence-notebook.ipynb")):
    item = artifact_by_path[relative]
    require(not path.is_symlink() and path.stat().st_size == item["sizeBytes"] and sha256(path) == item["sha256"], f"publication artifact binding failed: {relative}")
public_report = json.loads(report_path.read_bytes())


In [ ]:
trusted_files = {
    "templateSha256": bundle_dir / "perf-corpus-rlm-template.ipynb",
    "driverSha256": bundle_dir / "perf-corpus-rlm-analysis.py",
    "preregistrationSha256": bundle_dir / "perf-corpus-preregistration.json",
    "attemptLedgerSha256": input_dir / "perf-corpus-attempt-ledger.json",
    "rawManifestSha256": input_dir / "perf-corpus-raw-manifest.json",
}
for key, path in trusted_files.items():
    reject_symlink_components(path)
    require(not path.is_symlink() and not (path.stat().st_mode & 0o222) and sha256(path) == anchors[key], f"trusted binding failed: {key}")
raw_manifest = json.loads(trusted_files["rawManifestSha256"].read_bytes())
input_hashes = {}
for item in raw_manifest["reports"]:
    filename = item["filename"]
    require(pathlib.PurePosixPath(filename).name == filename, "non-contained report filename")
    path = input_dir / filename
    require(not path.is_symlink() and not (path.stat().st_mode & 0o222) and sha256(path) == item["sha256"], f"sealed report failed: {filename}")
    input_hashes[filename] = item["sha256"]
require(not (input_dir.stat().st_mode & 0o222), "sealed input directory is writable")
ledger = json.loads(trusted_files["attemptLedgerSha256"].read_bytes())
require(hashlib.sha256(ledger["captureId"].encode()).hexdigest() == anchors["captureIdSha256"], "capture identity binding failed")
closure = [manifest_path, report_path, notebook_path, *trusted_files.values(), *(input_dir / name for name in input_hashes)]
pre_hashes = {str(path): sha256(path) for path in closure}


In [ ]:
environment = {
    "GJC_PERF_CORPUS_BUNDLE_DIR": str(bundle_dir), "GJC_PERF_CORPUS_INPUT_DIR": str(input_dir), "GJC_PERF_CORPUS_OUTPUT_DIR": str(output_dir),
    "GJC_PERF_CORPUS_EXPECTED_GIT_SHA": anchors["measurementHead"], "GJC_PERF_CORPUS_EXPECTED_TREE_SHA": anchors["measurementTree"],
    "GJC_PERF_CORPUS_EXPECTED_CLOSURE_DIGEST": anchors["closureDigest"], "GJC_PERF_CORPUS_EXPECTED_WORKTREE_FINGERPRINT": anchors["worktreeFingerprint"],
    "GJC_PERF_CORPUS_EXPECTED_RUNTIME_CONTROL_IDENTITY": anchors["runtimeControlIdentity"], "GJC_PERF_CORPUS_EXPECTED_CAPTURE_ID": ledger["captureId"],
    "GJC_PERF_CORPUS_EXPECTED_SCHEDULE_DIGEST": anchors["scheduleDigest"], "GJC_PERF_CORPUS_EXPECTED_PROTOCOL_DIGEST": anchors["protocolDigest"],
    "GJC_PERF_CORPUS_TEMPLATE_SHA256": anchors["templateSha256"], "GJC_PERF_CORPUS_DRIVER_SHA256": anchors["driverSha256"],
    "GJC_PERF_CORPUS_PREREGISTRATION_SHA256": anchors["preregistrationSha256"], "GJC_PERF_CORPUS_ATTEMPT_LEDGER_SHA256": anchors["attemptLedgerSha256"],
    "GJC_PERF_CORPUS_RAW_MANIFEST_SHA256": anchors["rawManifestSha256"], "GJC_PERF_CORPUS_INPUT_MOUNT_READ_ONLY": "1",
}
os.environ.update(environment)
template = json.loads(trusted_files["templateSha256"].read_bytes())
code_cells = [cell for cell in template["cells"] if cell["cell_type"] == "code"]
require(len(code_cells) == 1, "unexpected template code-cell count")
displayed = []
def display(value): displayed.append(value)
exec(compile("".join(code_cells[0]["source"]), "<verified-template>", "exec"), {"display": display})
require(len(displayed) == 1, "template did not emit one terminal receipt")
terminal_receipt = displayed[0]


In [ ]:
result_path = output_dir / "perf-corpus-rlm-result.json"
markdown_path = output_dir / "perf-corpus-rlm-result.md"
result = json.loads(result_path.read_bytes())
output_entries = list(output_dir.iterdir())
require({path.name for path in output_entries} == {"perf-corpus-rlm-result.json", "perf-corpus-rlm-result.md"}, "unexpected output entry set")
require(all(not path.is_symlink() and path.is_file() for path in output_entries), "output entries must be non-symlink regular files")
require(sum(path.stat().st_size for path in output_entries) <= 1024 * 1024, "output exceeds one-mebibyte bound")
expected_report = {
    "schema": "gjc.perf-corpus-memory-evidence-report/1", "generatedAt": manifest["generatedAt"],
    "evidenceStatus": result["evidenceStatus"], "actionDecision": result["actionDecision"], "actionFamily": result["actionFamily"],
    "measurementHead": anchors["measurementHead"],
    "admission": {
        "short": {"required": result["admission"]["short"]["requiredAdmittedBlocks"], "admitted": result["admission"]["short"]["admittedBlocks"], "ratio": f"{result['admission']['short']['admittedBlocks']}/{result['admission']['short']['requiredAdmittedBlocks']}"},
        "soak": {"required": result["admission"]["soak"]["requiredAdmittedBlocks"], "admitted": result["admission"]["soak"]["admittedBlocks"], "ratio": f"{result['admission']['soak']['admittedBlocks']}/{result['admission']['soak']['requiredAdmittedBlocks']}"},
    },
    "surfaces": {},
    "p95Claim": {"status": result["claimPolicy"]["p95"]["status"], "reason": result["claimPolicy"]["p95"]["reason"], "method": result["claimPolicy"]["p95"]["method"]},
    "limitations": result["limitations"], "retentionAccess": manifest["retentionAccess"],
    "reproducibility": {"notebook": "artifacts/perf-corpus-memory-evidence-notebook.ipynb", "trustAnchors": "artifacts/perf-corpus-memory-evidence-manifest.json", "requiredExternalControls": ["OS-enforced network denial", "allowlisted sanitized environment", "read-only sealed-input closure", "fresh bounded output directory"], "canonicalResultSha256": sha256(result_path), "canonicalMarkdownSha256": sha256(markdown_path)},
}
for surface in ("agent-session", "tui"):
    observed = result["actionAnalysis"]["surfaces"][surface]
    expected_report["surfaces"][surface] = {
        "primaryEndpoint": {"medianBytesPerSecond": round(observed["primaryMedianBytesPerSecond"], 3), "bcaLowerBoundBytesPerSecond": round(observed["primaryBca"]["lower"], 3), "bcaUpperBoundBytesPerSecond": round(observed["primaryBca"]["upper"], 3), "count": observed["reportCount"], "confidenceLevel": observed["primaryBca"]["confidenceLevel"], "resamples": observed["primaryBca"]["resamples"], "seed": observed["primaryBca"]["seed"]},
        "slopeTheilSen": {"medianBytesPerSecond": round(observed["theilSenMedianBytesPerSecond"], 2), "count": observed["reportCount"]},
    }
require(public_report == expected_report, "published report differs from authenticated canonical projection")
expected_receipt = {"analysisSchema": result["schema"], "evidenceStatus": result["evidenceStatus"], "actionDecision": result["actionDecision"], "hashBindings": result["hashBindings"], "admissionTraceability": result["admissionTraceability"], "p95MethodReceipt": result["claimPolicy"]["p95"], "resultJsonPath": str(result_path), "resultMarkdownPath": str(markdown_path)}
require(terminal_receipt == expected_receipt, "terminal receipt mismatch")
for path in closure:
    require(sha256(path) == pre_hashes[str(path)], f"post-execution closure drift: {path.name}")
